# 06 · Motor de riesgo

## 1. Pwned Passwords con k-anonimato

In [2]:
import sys
sys.path.append("..")

import requests
from src.filtraciones import hash_sha1, veces_filtrada, URL_API

contrasena = "123456"
sha1 = hash_sha1(contrasena)
prefijo, sufijo = sha1[:5], sha1[5:]

print(f"Hash SHA-1:       {sha1}")
print(f"Se envía:         {prefijo}")
print(f"Se queda en el PC: {sufijo}")

respuesta = requests.get(URL_API + prefijo, headers={"Add-Padding": "true"})
print(f"Hashes que devolvió la API para ese prefijo: {len(respuesta.text.splitlines()):,}")

Hash SHA-1:       7C4A8D09CA3762AF61E59520943DC26494F8941B
Se envía:         7C4A8
Se queda en el PC: D09CA3762AF61E59520943DC26494F8941B
Hashes que devolvió la API para ese prefijo: 2,091


In [3]:
ejemplos = ["123456", "maria2005", "aaaaaaaaaaaaaaaa", "P@ssw0rd!",
            "AVYq1lDE4MgAZfNt", "Gato-Azul-Lluvia-77"]

for c in ejemplos:
    veces = veces_filtrada(c)
    print(f"{c:22} → {veces:,} veces" if veces is not None else f"{c:22} → no se pudo verificar")

123456                 → 210,463,629 veces
maria2005              → 21,496 veces
aaaaaaaaaaaaaaaa       → 23,337 veces
P@ssw0rd!              → 119,522 veces
AVYq1lDE4MgAZfNt       → 6 veces
Gato-Azul-Lluvia-77    → 0 veces


In [3]:
pip install zxcvbn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
2. Motor de riesgo

In [4]:
import pandas as pd
from src.riesgo import cargar_recursos, calcular_riesgo

recursos = cargar_recursos()

filas = []
for c in ejemplos:
    r = calcular_riesgo(c, recursos)
    filas.append({
        "Contraseña": c,
        "Puntaje": r["puntaje"],
        "Nivel": r["nivel"],
        "Veces filtrada": r["veces_filtrada"],
        "zxcvbn (0-4)": r["zxcvbn_score"],
        "Tiempo para adivinarla": r["tiempo_para_adivinarla"],
        "Prob. modelo": round(r["probabilidad_modelo"], 2),
        "En RockYou": r["en_rockyou"],
    })

pd.DataFrame(filas)

,Contraseña,Puntaje,Nivel,Veces filtrada,zxcvbn (0-4),Tiempo para adivinarla,Prob. modelo,En RockYou
0,123456,99,Alto,210463629,0,less than a second,0.96,True
1,maria2005,80,Alto,21496,1,2 seconds,0.67,True
2,aaaaaaaaaaaaaaaa,77,Alto,23337,0,less than a second,0.25,True
3,P@ssw0rd!,70,Alto,119522,1,1 second,0.24,False
4,AVYq1lDE4MgAZfNt,70,Alto,6,4,centuries,0.00,False
5,Gato-Azul-Lluvia-77,0,Bajo,0,4,centuries,0.00,False


In [5]:
for c in ["Carlos#1998mx", "colombia2026tq", "Mi-Perro-Toby-2019"]:
    r = calcular_riesgo(c, recursos)
    print(f"{c:22} → {r['puntaje']:3} {r['nivel']}")

Carlos#1998mx          →   8 Bajo
colombia2026tq         →   9 Bajo
Mi-Perro-Toby-2019     →   0 Bajo


In [6]:
for c in ["tortuga88", "girasol07", "Maleta2029", "mesa1234"]:
    r = calcular_riesgo(c, recursos)
    print(f"{c:22} → {r['puntaje']:3} {r['nivel']} | filtrada: {r['veces_filtrada']}")

tortuga88              →  73 Alto | filtrada: 1496
girasol07              →  70 Alto | filtrada: 157
Maleta2029             →  30 Medio | filtrada: 0
mesa1234               →  81 Alto | filtrada: 7973


In [7]:
for c in ["Maleta2029", "Carlos#1998mx", "colombia2026tq", "Mi-Perro-Toby-2019", "Gato-Azul-Lluvia-77"]:
    r = calcular_riesgo(c, recursos)
    print(f"{c:22} → {r['puntaje']:3} {r['nivel']}")

Maleta2029             →  30 Medio
Carlos#1998mx          →   8 Bajo
colombia2026tq         →   9 Bajo
Mi-Perro-Toby-2019     →   0 Bajo
Gato-Azul-Lluvia-77    →   0 Bajo


In [8]:
calcular_riesgo("Maleta2029", recursos)["razones"]

['Es corta y sigue un patrón predecible (palabra + números o un año).']

- Se agregó una regla: una contraseña de menos de 12 caracteres con un patrón predecible
  ("palabra + números" o un año) tiene riesgo al menos medio (30). Con esto, "Maleta2029"
  pasa de bajo (10) a medio (30), sin afectar contraseñas largas como "Mi-Perro-Toby-2019".